# Audio Filtering Techniques Comparison

Test 7 individual filtering techniques + 4 combinations on Bulgarian audio.
Transcribe each with **Groq API** (free Whisper large-v3) and compare results.

**Input:** `data/shorter.mp4` (first 3 min)

| # | Technique | Goal |
|---|-----------|------|
| 0 | Baseline | No filtering (reference) |
| 1 | Noise Reduction | Remove background hiss |
| 2 | VAD Trimming | Remove silence (prevent hallucinations) |
| 3 | Loudness Normalization | Optimal input level |
| 4 | High-Pass + De-Essing | Remove rumble + tame sibilants |
| 5 | Dynamic Range Compression | Even out speaker volumes |
| 6 | Bandpass + Pre-Emphasis | Focus on speech frequencies |
| 7 | Adaptive VAD Filtering | Gentle NR on speech, silence non-speech regions |
| A | Noise + Normalization | "Safe" combo |
| B | HP + Noise + Comp + Norm | "Studio" chain |
| C | Full Pipeline | All techniques stacked |
| D | HP + Adaptive VAD + Norm | Smart speech-aware combo |

## 1. Setup & Imports

In [1]:
# !pip install noisereduce pyloudnorm groq

import json
import os
import subprocess
import tempfile
import time
from pathlib import Path

import numpy as np
import torch
from scipy.io import wavfile
from scipy.signal import butter, sosfilt

import noisereduce as nr
import pyloudnorm as pyln
from groq import Groq

# Fix ffmpeg
try:
    subprocess.run(["ffmpeg", "-version"], capture_output=True, check=True)
except (subprocess.CalledProcessError, OSError):
    os.environ["PATH"] = "/usr/bin:" + os.environ.get("PATH", "")

# Groq API
GROQ_KEY = "YOUR_GROQ_API_KEY"
groq_client = Groq(api_key=GROQ_KEY)

# Paths
PROJECT_ROOT = Path(".").resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output" / "filtering_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_FILE = DATA_DIR / "234602045" / "pe202410_real_100774_234602045_20241027_200420_71.mp4"
WAV_FILE = OUTPUT_DIR / "election_6to12min.wav"
SR = 16000
EXTRACT_START = 360   # 6:00
EXTRACT_DURATION = 360  # 6 minutes (6:00 to 12:00)

print(f"Video: {VIDEO_FILE.name} ({VIDEO_FILE.stat().st_size / 1024 / 1024:.0f} MB)")
print(f"Extract: min {EXTRACT_START//60} to {(EXTRACT_START+EXTRACT_DURATION)//60}")
print(f"Groq key: ...{GROQ_KEY[-4:]}")

Video: pe202410_real_100774_234602045_20241027_200420_71.mp4 (1053 MB)
Extract: min 6 to 12
Groq key: ...de3Z


In [2]:
# Extract audio from min 6 to min 12
if not WAV_FILE.exists():
    print(f"Extracting min {EXTRACT_START//60}-{(EXTRACT_START+EXTRACT_DURATION)//60} from {VIDEO_FILE.name}...")
    subprocess.run(
        ["ffmpeg", "-ss", str(EXTRACT_START), "-i", str(VIDEO_FILE),
         "-t", str(EXTRACT_DURATION),
         "-vn", "-acodec", "pcm_s16le", "-ar", str(SR), "-ac", "1",
         "-y", str(WAV_FILE)],
        capture_output=True, check=True,
    )
    print(f"Saved: {WAV_FILE.name} ({WAV_FILE.stat().st_size / 1024 / 1024:.1f} MB)")
else:
    print(f"WAV exists: {WAV_FILE.name}")

# Load audio
_, audio_int16 = wavfile.read(str(WAV_FILE))
audio = audio_int16.astype(np.float32) / 32768.0
duration = len(audio) / SR
print(f"Duration: {duration:.1f}s | Samples: {len(audio):,} | SR: {SR}")

# Helper: save float32 audio to WAV
def save_wav(audio_data, path):
    clipped = np.clip(audio_data, -1.0, 1.0)
    wavfile.write(str(path), SR, (clipped * 32767).astype(np.int16))

# Helper: transcribe with Groq API
def transcribe_groq(audio_data, label):
    wav_path = OUTPUT_DIR / f"filtered_{label}.wav"
    save_wav(audio_data, wav_path)
    
    t0 = time.time()
    with open(wav_path, "rb") as f:
        response = groq_client.audio.transcriptions.create(
            file=(wav_path.name, f.read()),
            model="whisper-large-v3",
            language="bg",
            response_format="verbose_json",
            temperature=0.0,
        )
    elapsed = time.time() - t0
    
    result = response.model_dump()
    text = result.get("text", "")
    segments = result.get("segments", [])
    audio_dur = len(audio_data) / SR
    
    print(f"  [{label}] {elapsed:.1f}s API | {len(segments)} segments | {len(text)} chars | {audio_dur:.1f}s audio")
    for seg in segments[:3]:
        print(f"    [{seg['start']:.1f}-{seg['end']:.1f}s] {seg['text'].strip()}")
    if len(segments) > 3:
        print(f"    ... ({len(segments) - 3} more)")
    
    return {"label": label, "text": text, "segments": segments,
            "chars": len(text), "num_segments": len(segments),
            "audio_duration": audio_dur, "api_time": elapsed}

results = {}
print("\nReady to test filtering techniques.")

WAV exists: election_6to12min.wav
Duration: 360.0s | Samples: 5,760,000 | SR: 16000

Ready to test filtering techniques.


## 2. Baseline — No Filtering

In [3]:
print("=== BASELINE (no filtering) ===")
results["0_baseline"] = transcribe_groq(audio, "0_baseline")

=== BASELINE (no filtering) ===


  [0_baseline] 4.1s API | 111 segments | 2132 chars | 360.0s audio
    [0.0-14.5s] Остройството не е позиционирано правилно.
    [15.0-17.9s] Завъртете го така, че физическите бутони да бъдат отгоре.
    [17.9-25.1s] А това е ОК. Не е изглазило.
    ... (108 more)


## 3. Technique 1 — Noise Reduction (noisereduce)
Spectral gating: estimates noise profile and removes steady-state background noise.

In [4]:
print("=== TECHNIQUE 1: Noise Reduction ===")
audio_nr = nr.reduce_noise(y=audio, sr=SR, prop_decrease=0.6, stationary=True)
print(f"  RMS before: {np.sqrt(np.mean(audio**2)):.4f} | after: {np.sqrt(np.mean(audio_nr**2)):.4f}")
results["1_noise_reduction"] = transcribe_groq(audio_nr, "1_noise_reduction")

=== TECHNIQUE 1: Noise Reduction ===


  RMS before: 0.0350 | after: 0.0256


  [1_noise_reduction] 4.2s API | 111 segments | 2001 chars | 360.0s audio
    [0.0-14.5s] Остройството не е позиционирано правилно.
    [15.0-17.9s] Завъртете го така, че физическите бутони да бъдат отгоре.
    [17.9-25.1s] А това е ОК. Не е изглазило.
    ... (108 more)


## 4. Technique 2 — VAD Trimming (Silero VAD)
Remove non-speech segments. Whisper hallucinates on silence.

In [5]:
print("=== TECHNIQUE 2: VAD Trimming (Silero) ===")
vad_model, vad_utils = torch.hub.load("snakers4/silero-vad", "silero_vad")
get_speech_timestamps = vad_utils[0]

audio_tensor = torch.from_numpy(audio).float()
speech_timestamps = get_speech_timestamps(audio_tensor, vad_model, sampling_rate=SR)

# Extract and concatenate speech segments
speech_chunks = []
for ts in speech_timestamps:
    speech_chunks.append(audio[ts["start"]:ts["end"]])

audio_vad = np.concatenate(speech_chunks) if speech_chunks else audio
speech_ratio = len(audio_vad) / len(audio) * 100
print(f"  Speech segments: {len(speech_timestamps)} | Duration: {len(audio_vad)/SR:.1f}s ({speech_ratio:.0f}% of original)")
results["2_vad_trimming"] = transcribe_groq(audio_vad, "2_vad_trimming")

=== TECHNIQUE 2: VAD Trimming (Silero) ===


Using cache found in /home/airsrg/.cache/torch/hub/snakers4_silero-vad_master


  Speech segments: 44 | Duration: 84.2s (23% of original)


  [2_vad_trimming] 3.9s API | 37 segments | 806 chars | 84.2s audio
    [0.0-1.3s] позиционирано правилно.
    [1.8-3.4s] Завъртете го така, че физическите
    [3.4-4.7s] бутони да бъдат отгоре.
    ... (34 more)


## 5. Technique 3 — Loudness Normalization (pyloudnorm)
Normalize to -16 LUFS (standard streaming loudness).

In [6]:
print("=== TECHNIQUE 3: Loudness Normalization ===")
meter = pyln.Meter(SR)
loudness = meter.integrated_loudness(audio)
audio_norm = pyln.normalize.loudness(audio, loudness, -16.0)
print(f"  Loudness before: {loudness:.1f} LUFS | after: -16.0 LUFS")
results["3_loudness_norm"] = transcribe_groq(audio_norm, "3_loudness_norm")

=== TECHNIQUE 3: Loudness Normalization ===


/home/airsrg/mambaforge/lib/python3.10/site-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")


  Loudness before: -25.3 LUFS | after: -16.0 LUFS


  [3_loudness_norm] 5.1s API | 125 segments | 2416 chars | 360.0s audio
    [0.0-14.5s] Остройството не е позиционирано правилно.
    [15.0-17.0s] Завъртете го така, че физическите бутони
    [17.0-17.9s] да бъдат отгоре.
    ... (122 more)


## 6. Technique 4 — High-Pass Filter + De-Essing
Butterworth high-pass at 80Hz removes rumble/hum. Gentle shelf above 5kHz tames harsh sibilants.

In [7]:
print("=== TECHNIQUE 4: High-Pass + De-Essing ===")

# High-pass filter at 80Hz
sos_hp = butter(5, 80, btype="highpass", fs=SR, output="sos")
audio_hp = sosfilt(sos_hp, audio)

# Gentle de-essing: low-pass shelf above 5kHz (reduce by mixing with low-passed version)
sos_lp = butter(3, 5000, btype="lowpass", fs=SR, output="sos")
audio_lp = sosfilt(sos_lp, audio_hp)
# Mix: keep 70% of high frequencies (gentle de-essing, not aggressive)
audio_deessed = audio_lp + 0.7 * (audio_hp - audio_lp)

print(f"  High-pass: 80Hz cutoff | De-essing: 5kHz shelf (-3dB)")
results["4_highpass_deess"] = transcribe_groq(audio_deessed, "4_highpass_deess")

=== TECHNIQUE 4: High-Pass + De-Essing ===
  High-pass: 80Hz cutoff | De-essing: 5kHz shelf (-3dB)


  [4_highpass_deess] 3.3s API | 120 segments | 2207 chars | 360.0s audio
    [0.0-14.5s] Остройството не е позиционирано правилно.
    [15.0-17.9s] Завъртете го така, че физическите бутони да бъдат отгоре.
    [17.9-25.1s] А това е ОК. Не е изглазило.
    ... (117 more)


## 7. Technique 5 — Dynamic Range Compression
Soft compressor to even out volume differences between speakers.

In [8]:
print("=== TECHNIQUE 5: Dynamic Range Compression ===")

def compress_audio(audio_data, threshold_db=-20, ratio=3.0, attack_ms=5, release_ms=50):
    """Simple soft-knee dynamic range compressor."""
    threshold = 10 ** (threshold_db / 20)
    attack_samples = int(SR * attack_ms / 1000)
    release_samples = int(SR * release_ms / 1000)
    
    envelope = np.abs(audio_data)
    # Smooth envelope with attack/release
    smoothed = np.zeros_like(envelope)
    for i in range(1, len(envelope)):
        if envelope[i] > smoothed[i-1]:
            coeff = 1.0 - np.exp(-1.0 / max(attack_samples, 1))
        else:
            coeff = 1.0 - np.exp(-1.0 / max(release_samples, 1))
        smoothed[i] = smoothed[i-1] + coeff * (envelope[i] - smoothed[i-1])
    
    # Apply compression
    gain = np.ones_like(smoothed)
    above = smoothed > threshold
    gain[above] = threshold * (smoothed[above] / threshold) ** (1.0 / ratio) / smoothed[above]
    
    compressed = audio_data * gain
    # Makeup gain to match original RMS
    rms_orig = np.sqrt(np.mean(audio_data ** 2))
    rms_comp = np.sqrt(np.mean(compressed ** 2))
    if rms_comp > 0:
        compressed *= rms_orig / rms_comp
    return compressed

audio_comp = compress_audio(audio, threshold_db=-20, ratio=3.0)
print(f"  Threshold: -20dB | Ratio: 3:1")
print(f"  Peak before: {np.max(np.abs(audio)):.3f} | after: {np.max(np.abs(audio_comp)):.3f}")
results["5_compression"] = transcribe_groq(audio_comp, "5_compression")

=== TECHNIQUE 5: Dynamic Range Compression ===


  Threshold: -20dB | Ratio: 3:1
  Peak before: 1.000 | after: 1.237


  [5_compression] 3.7s API | 119 segments | 2093 chars | 360.0s audio
    [0.0-14.5s] Остройството не е позиционирано правилно.
    [15.0-17.0s] Завъртете го така, че физическите бутони
    [17.0-17.9s] да бъдат отгоре.
    ... (116 more)


## 8. Technique 6 — Bandpass + Pre-Emphasis
Bandpass filter (300Hz-7kHz) focuses on speech frequencies. Pre-emphasis boosts high frequencies for consonant clarity.

In [9]:
print("=== TECHNIQUE 6: Bandpass + Pre-Emphasis ===")

# Bandpass: 300Hz - 7kHz (speech range)
sos_bp = butter(4, [300, 7000], btype="bandpass", fs=SR, output="sos")
audio_bp = sosfilt(sos_bp, audio)

# Pre-emphasis: boost high frequencies (standard ASR preprocessing)
# y[n] = x[n] - 0.97 * x[n-1]
pre_emphasis_coeff = 0.97
audio_preemph = np.append(audio_bp[0], audio_bp[1:] - pre_emphasis_coeff * audio_bp[:-1])

print(f"  Bandpass: 300Hz-7kHz | Pre-emphasis: {pre_emphasis_coeff}")
results["6_bandpass_preemph"] = transcribe_groq(audio_preemph, "6_bandpass_preemph")

=== TECHNIQUE 6: Bandpass + Pre-Emphasis ===
  Bandpass: 300Hz-7kHz | Pre-emphasis: 0.97


  [6_bandpass_preemph] 3.7s API | 79 segments | 1641 chars | 360.0s audio
    [0.0-14.5s] Остройството не е позиционирано правилно.
    [15.0-17.9s] Завъртете го така, че физическите бутони да бъдат отгоре.
    [30.0-34.0s] Два и шеста, две и шеста и двадесет.
    ... (76 more)


## 9. Combo A — Noise Reduction + Loudness Normalization
The "safe" combination: clean noise first, then normalize loudness. Low risk of artifacts.

In [10]:
print("=== COMBO A: Noise Reduction + Loudness Normalization ===")
combo_a = nr.reduce_noise(y=audio, sr=SR, prop_decrease=0.6, stationary=True)
loudness_a = meter.integrated_loudness(combo_a)
combo_a = pyln.normalize.loudness(combo_a, loudness_a, -16.0)
print(f"  Pipeline: noise_reduce -> loudness_norm(-16 LUFS)")
results["A_nr_norm"] = transcribe_groq(combo_a, "A_nr_norm")

=== COMBO A: Noise Reduction + Loudness Normalization ===


/home/airsrg/mambaforge/lib/python3.10/site-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")


  Pipeline: noise_reduce -> loudness_norm(-16 LUFS)


  [A_nr_norm] 4.2s API | 119 segments | 2210 chars | 360.0s audio
    [0.0-14.5s] Остройството не е позиционирано правилно.
    [15.0-17.0s] Завъртете го така, че физическите бутони
    [17.0-17.9s] да бъдат отгоре.
    ... (116 more)


## 10. Combo B — High-Pass + Noise Reduction + Compression + Normalization
The "studio" chain — full signal processing without VAD.

In [11]:
print("=== COMBO B: HP + Noise Reduction + Compression + Normalization ===")
# 1. High-pass 80Hz
combo_b = sosfilt(butter(5, 80, btype="highpass", fs=SR, output="sos"), audio)
# 2. Noise reduction
combo_b = nr.reduce_noise(y=combo_b, sr=SR, prop_decrease=0.6, stationary=True)
# 3. Dynamic range compression
combo_b = compress_audio(combo_b, threshold_db=-20, ratio=3.0)
# 4. Loudness normalization
loudness_b = meter.integrated_loudness(combo_b)
combo_b = pyln.normalize.loudness(combo_b, loudness_b, -16.0)
print(f"  Pipeline: HP(80Hz) -> noise_reduce -> compress(3:1) -> loudness_norm")
results["B_studio"] = transcribe_groq(combo_b, "B_studio")

=== COMBO B: HP + Noise Reduction + Compression + Normalization ===


/home/airsrg/mambaforge/lib/python3.10/site-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")


  Pipeline: HP(80Hz) -> noise_reduce -> compress(3:1) -> loudness_norm


  [B_studio] 4.0s API | 112 segments | 2136 chars | 360.0s audio
    [0.0-14.5s] Остройството не е позиционирано правилно.
    [15.0-17.0s] Завъртете го така, че физическите бутони
    [17.0-17.9s] да бъдат отгоре.
    ... (109 more)


## 11. Combo C — Full Pipeline (all techniques)
HP → Noise reduction → Compression → Normalization → VAD trim. The "kitchen sink".

In [12]:
print("=== COMBO C: Full Pipeline ===")
# 1. High-pass 80Hz
combo_c = sosfilt(butter(5, 80, btype="highpass", fs=SR, output="sos"), audio)
# 2. Noise reduction
combo_c = nr.reduce_noise(y=combo_c, sr=SR, prop_decrease=0.6, stationary=True)
# 3. Dynamic range compression
combo_c = compress_audio(combo_c, threshold_db=-20, ratio=3.0)
# 4. Loudness normalization
loudness_c = meter.integrated_loudness(combo_c)
combo_c = pyln.normalize.loudness(combo_c, loudness_c, -16.0)
# 5. VAD trimming
combo_c_tensor = torch.from_numpy(combo_c).float()
speech_ts_c = get_speech_timestamps(combo_c_tensor, vad_model, sampling_rate=SR)
chunks_c = [combo_c[ts["start"]:ts["end"]] for ts in speech_ts_c]
combo_c_vad = np.concatenate(chunks_c) if chunks_c else combo_c
print(f"  Pipeline: HP -> NR -> Comp -> Norm -> VAD ({len(combo_c_vad)/SR:.1f}s)")
results["C_full"] = transcribe_groq(combo_c_vad, "C_full")

=== COMBO C: Full Pipeline ===


/home/airsrg/mambaforge/lib/python3.10/site-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")


  Pipeline: HP -> NR -> Comp -> Norm -> VAD (264.1s)


  [C_full] 2.6s API | 92 segments | 1659 chars | 264.1s audio
    [0.0-3.0s] Стройството не е позиционирано правилно.
    [3.0-6.0s] Завъртете го така, че физическите бутони да бъдат отгоре.
    [6.0-8.0s] Това не е изгрязало.
    ... (89 more)


## 12. Technique 7 — Adaptive VAD Filtering (speech-aware)

Instead of just trimming silence, this technique **adapts filtering intensity based on whether speech is present**:
- **Speech detected** → gentle filtering (preserve quality, light noise reduction)
- **No speech** → aggressive attenuation (silence the noise floor)

This keeps the original timeline intact (no concatenation) while cleaning non-speech regions.

In [13]:
print("=== TECHNIQUE 7: Adaptive VAD Filtering ===")

# Get VAD speech timestamps (reuse model from earlier)
audio_t = torch.from_numpy(audio).float()
speech_ts = get_speech_timestamps(audio_t, vad_model, sampling_rate=SR)

# Build a speech mask (True = speech, False = non-speech)
speech_mask = np.zeros(len(audio), dtype=bool)
for ts in speech_ts:
    # Add small padding around speech (50ms) to avoid cutting edges
    pad = int(0.05 * SR)
    start = max(0, ts["start"] - pad)
    end = min(len(audio), ts["end"] + pad)
    speech_mask[start:end] = True

speech_pct = speech_mask.sum() / len(speech_mask) * 100
print(f"  Speech: {speech_pct:.0f}% of audio | {len(speech_ts)} segments")

# Speech regions: gentle noise reduction (preserve quality)
audio_speech = nr.reduce_noise(y=audio, sr=SR, prop_decrease=0.3, stationary=True)

# Non-speech regions: aggressive attenuation (kill the noise)
audio_adaptive = audio.copy()
audio_adaptive[speech_mask] = audio_speech[speech_mask]       # gentle NR on speech
audio_adaptive[~speech_mask] = audio[~speech_mask] * 0.02     # nearly silence non-speech

print(f"  Speech regions: light NR (prop_decrease=0.3)")
print(f"  Non-speech regions: attenuated to 2%")
results["7_adaptive_vad"] = transcribe_groq(audio_adaptive, "7_adaptive_vad")

=== TECHNIQUE 7: Adaptive VAD Filtering ===


  Speech: 25% of audio | 44 segments


  Speech regions: light NR (prop_decrease=0.3)
  Non-speech regions: attenuated to 2%


  [7_adaptive_vad] 3.9s API | 42 segments | 903 chars | 360.0s audio
    [0.0-14.5s] Позиционирано правилно.
    [15.0-17.0s] Завъртете го така, че физическите бутони
    [17.0-17.9s] да бъдат отгоре.
    ... (39 more)


## 13. Combo D — HP + Adaptive VAD + Normalization
High-pass to clean low-end, then adaptive VAD filtering (gentle NR on speech, silence non-speech), then normalize.

In [14]:
print("=== COMBO D: HP + Adaptive VAD + Normalization ===")

# 1. High-pass 80Hz
combo_d = sosfilt(butter(5, 80, btype="highpass", fs=SR, output="sos"), audio)

# 2. Adaptive VAD filtering
combo_d_speech = nr.reduce_noise(y=combo_d, sr=SR, prop_decrease=0.3, stationary=True)
combo_d_out = combo_d.copy()
combo_d_out[speech_mask] = combo_d_speech[speech_mask]     # gentle NR on speech
combo_d_out[~speech_mask] = combo_d[~speech_mask] * 0.02   # silence non-speech

# 3. Loudness normalization
loudness_d = meter.integrated_loudness(combo_d_out)
combo_d_out = pyln.normalize.loudness(combo_d_out, loudness_d, -16.0)

print(f"  Pipeline: HP(80Hz) -> adaptive_vad(NR speech / silence gaps) -> norm(-16 LUFS)")
results["D_hp_adaptive_norm"] = transcribe_groq(combo_d_out, "D_hp_adaptive_norm")

=== COMBO D: HP + Adaptive VAD + Normalization ===


/home/airsrg/mambaforge/lib/python3.10/site-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")


  Pipeline: HP(80Hz) -> adaptive_vad(NR speech / silence gaps) -> norm(-16 LUFS)


  [D_hp_adaptive_norm] 2.7s API | 30 segments | 742 chars | 360.0s audio
    [0.0-14.5s] Остройството не е позиционирано правилно.
    [15.0-17.9s] Завъртете го така, че физическите бутони да бъдат отгоре.
    [30.0-60.0s] Два, три, шест, девет, девет.
    ... (27 more)


## 14. Volume Recovery Techniques

The 25-43s section has background speech that's 2.6x quieter (8.2 dB) than the main speakers. These techniques aggressively boost quiet speech to catch what's being said.

In [15]:
print("=== TECHNIQUE 8: AGC (Automatic Gain Control) ===")
# Sliding window AGC — normalizes volume per short window
# This brings quiet background speech up to the same level as loud speech

def agc(audio_data, sr, window_ms=500, target_rms=0.08):
    """Automatic Gain Control with sliding window."""
    window_samples = int(sr * window_ms / 1000)
    output = np.zeros_like(audio_data)
    
    for i in range(0, len(audio_data), window_samples // 2):  # 50% overlap
        end = min(i + window_samples, len(audio_data))
        chunk = audio_data[i:end]
        rms = np.sqrt(np.mean(chunk**2))
        if rms > 0.001:  # avoid amplifying silence
            gain = min(target_rms / rms, 10.0)  # cap at 10x boost
        else:
            gain = 1.0
        # Smooth transition
        for j in range(i, end):
            if output[j] == 0:
                output[j] = audio_data[j] * gain
            else:
                output[j] = (output[j] + audio_data[j] * gain) / 2  # blend overlap
    
    return np.clip(output, -1.0, 1.0)

audio_agc = agc(audio, SR, window_ms=500, target_rms=0.08)
print(f"  Window: 500ms | Target RMS: 0.08")
print(f"  RMS quiet section before: {np.sqrt(np.mean(audio[25*SR:43*SR]**2)):.5f}")
print(f"  RMS quiet section after:  {np.sqrt(np.mean(audio_agc[25*SR:43*SR]**2)):.5f}")
results["8_agc"] = transcribe_groq(audio_agc, "8_agc")

=== TECHNIQUE 8: AGC (Automatic Gain Control) ===


  Window: 500ms | Target RMS: 0.08
  RMS quiet section before: 0.02127
  RMS quiet section after:  0.07627


  [8_agc] 4.7s API | 115 segments | 1993 chars | 360.0s audio
    [0.0-14.5s] Остройството не е позиционирано правилно.
    [15.0-17.0s] Завъртете го така, че физическите бутони
    [17.0-17.9s] да бъдат отгоре.
    ... (112 more)


In [16]:
print("=== TECHNIQUE 9: Speech-Band Boost + Aggressive Compression ===")
# Boost the 300Hz-3.5kHz band (core speech frequencies) by 6dB
# Then apply aggressive compression to bring up quiet parts

from scipy.signal import butter, sosfilt

# Speech-band boost: bandpass 300-3500Hz, mix boosted version with original
sos_speech = butter(4, [300, 3500], btype="bandpass", fs=SR, output="sos")
speech_band = sosfilt(sos_speech, audio)
# Boost speech band by 6dB (2x) and mix back
audio_boosted = audio + speech_band * 1.0  # effectively +6dB in speech band

# Aggressive compression (ratio 6:1, threshold -25dB)
audio_boost_comp = compress_audio(audio_boosted, threshold_db=-25, ratio=6.0)

# Normalize
loudness_bc = meter.integrated_loudness(audio_boost_comp)
audio_boost_comp = pyln.normalize.loudness(audio_boost_comp, loudness_bc, -14.0)  # louder target

print(f"  Speech band boost: +6dB (300-3500Hz)")
print(f"  Compression: 6:1 ratio, -25dB threshold")
print(f"  Normalization: -14 LUFS (louder than standard)")
results["9_boost_compress"] = transcribe_groq(audio_boost_comp, "9_boost_compress")

=== TECHNIQUE 9: Speech-Band Boost + Aggressive Compression ===


/home/airsrg/mambaforge/lib/python3.10/site-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")


  Speech band boost: +6dB (300-3500Hz)
  Compression: 6:1 ratio, -25dB threshold
  Normalization: -14 LUFS (louder than standard)


  [9_boost_compress] 4.0s API | 129 segments | 2413 chars | 360.0s audio
    [0.0-14.5s] Остройството не е позиционирано правилно.
    [15.0-17.0s] Завъртете го така, че физическите бутони
    [17.0-17.9s] да бъдат отгоре.
    ... (126 more)


In [17]:
print("=== TECHNIQUE 10: Multi-Band AGC ===")
# Split audio into 3 frequency bands, apply AGC to each independently
# This boosts quiet speech without distorting loud parts

# Band 1: Low (80-300Hz) — rumble, male fundamentals
sos_lo = butter(4, [80, 300], btype="bandpass", fs=SR, output="sos")
band_lo = sosfilt(sos_lo, audio)

# Band 2: Mid (300-3500Hz) — core speech
sos_mid = butter(4, [300, 3500], btype="bandpass", fs=SR, output="sos")
band_mid = sosfilt(sos_mid, audio)

# Band 3: High (3500-7500Hz) — consonants, sibilants
sos_hi = butter(4, [3500, 7500], btype="bandpass", fs=SR, output="sos")
band_hi = sosfilt(sos_hi, audio)

# Apply AGC to each band separately (mid band gets most boost)
band_lo_agc = agc(band_lo, SR, window_ms=500, target_rms=0.02)
band_mid_agc = agc(band_mid, SR, window_ms=300, target_rms=0.07)  # strongest boost
band_hi_agc = agc(band_hi, SR, window_ms=300, target_rms=0.03)

# Recombine
audio_multiband = band_lo_agc + band_mid_agc + band_hi_agc
loudness_mb = meter.integrated_loudness(audio_multiband)
audio_multiband = pyln.normalize.loudness(audio_multiband, loudness_mb, -16.0)

print(f"  3 bands: Low(80-300Hz), Mid(300-3500Hz), High(3500-7500Hz)")
print(f"  Per-band AGC with speech-focused mid boost")
results["10_multiband_agc"] = transcribe_groq(audio_multiband, "10_multiband_agc")

=== TECHNIQUE 10: Multi-Band AGC ===


/home/airsrg/mambaforge/lib/python3.10/site-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")


  3 bands: Low(80-300Hz), Mid(300-3500Hz), High(3500-7500Hz)
  Per-band AGC with speech-focused mid boost


  [10_multiband_agc] 7.3s API | 130 segments | 2538 chars | 360.0s audio
    [0.0-14.5s] Остройството не е позиционирано правилно.
    [15.0-17.0s] Завъртете го така, че физическите бутони
    [17.0-17.9s] да бъдат отгоре.
    ... (127 more)


In [18]:
print("=== COMBO E: NR + AGC + Speech Boost + Normalization ===")
# Best combo for catching quiet background speech:
# 1. Light noise reduction (preserve quiet speech, remove hiss)
# 2. AGC to equalize volume across time
# 3. Speech-band boost for clarity
# 4. Normalize

# 1. Light NR
combo_e = nr.reduce_noise(y=audio, sr=SR, prop_decrease=0.4, stationary=True)

# 2. AGC
combo_e = agc(combo_e, SR, window_ms=400, target_rms=0.07)

# 3. Speech-band boost (+4dB in 300-3500Hz)
speech_band_e = sosfilt(butter(4, [300, 3500], btype="bandpass", fs=SR, output="sos"), combo_e)
combo_e = combo_e + speech_band_e * 0.6

# 4. Normalize
loudness_e = meter.integrated_loudness(combo_e)
combo_e = pyln.normalize.loudness(combo_e, loudness_e, -14.0)

print(f"  Pipeline: Light NR -> AGC(400ms) -> Speech boost(+4dB) -> Norm(-14 LUFS)")
print(f"  Quiet section RMS: {np.sqrt(np.mean(audio[25*SR:43*SR]**2)):.5f} -> {np.sqrt(np.mean(combo_e[25*SR:43*SR]**2)):.5f}")
results["E_nr_agc_boost"] = transcribe_groq(combo_e, "E_nr_agc_boost")

=== COMBO E: NR + AGC + Speech Boost + Normalization ===


/home/airsrg/mambaforge/lib/python3.10/site-packages/pyloudnorm/normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")


  Pipeline: Light NR -> AGC(400ms) -> Speech boost(+4dB) -> Norm(-14 LUFS)
  Quiet section RMS: 0.02127 -> 0.19973


  [E_nr_agc_boost] 9.0s API | 125 segments | 2350 chars | 360.0s audio
    [0.0-14.5s] Остройството не е позиционирано правилно.
    [15.0-17.0s] Завъртете го така, че физическите бутони
    [17.0-17.9s] да бъдат отгоре.
    ... (122 more)


## 15. Compare All Results (including volume recovery)

In [19]:
import re

# Summary table
print(f"{'Technique':<30} {'Segs':>5} {'Chars':>6} {'Duration':>9} {'API':>6} {'Latin%':>7} {'Verdict'}")
print("=" * 85)
for key in sorted(results.keys()):
    r = results[key]
    text = r["text"]
    cyrillic = len(re.findall(r'[а-яА-ЯёЁ]', text))
    latin = len(re.findall(r'[a-zA-Z]', text))
    total = cyrillic + latin
    latin_pct = (latin / total * 100) if total > 0 else 0
    verdict = "HALLUC" if latin_pct > 5 else "clean" if latin_pct < 1 else "minor"
    print(f"{r['label']:<30} {r['num_segments']:>5} {r['chars']:>6} {r['audio_duration']:>8.1f}s {r['api_time']:>5.1f}s {latin_pct:>6.1f}% {verdict}")

# Baseline comparison
baseline_chars = results["0_baseline"]["chars"]
print(f"\n{'='*85}")
print(f"Baseline: {baseline_chars} chars (with hallucinations)")
print(f"\nChar difference from baseline:")
for key in sorted(results.keys()):
    r = results[key]
    diff = r["chars"] - baseline_chars
    pct = (diff / baseline_chars * 100) if baseline_chars > 0 else 0
    marker = "+" if diff > 0 else ""
    print(f"  {r['label']:<30} {marker}{diff:>5} chars ({marker}{pct:.1f}%)")

Technique                       Segs  Chars  Duration    API  Latin% Verdict
0_baseline                       111   2132    360.0s   4.1s    0.0% clean
10_multiband_agc                 130   2538    360.0s   7.3s    0.0% clean
1_noise_reduction                111   2001    360.0s   4.2s    0.0% clean
2_vad_trimming                    37    806     84.2s   3.9s    0.0% clean
3_loudness_norm                  125   2416    360.0s   5.1s    0.8% clean
4_highpass_deess                 120   2207    360.0s   3.3s    0.0% clean
5_compression                    119   2093    360.0s   3.7s    0.0% clean
6_bandpass_preemph                79   1641    360.0s   3.7s    0.0% clean
7_adaptive_vad                    42    903    360.0s   3.9s    3.8% minor
8_agc                            115   1993    360.0s   4.7s    0.0% clean
9_boost_compress                 129   2413    360.0s   4.0s    0.0% clean
A_nr_norm                        119   2210    360.0s   4.2s    0.4% clean
B_studio               

In [20]:
# Full text comparison — print each transcript for manual review
for key in sorted(results.keys()):
    r = results[key]
    print(f"\n{'='*70}")
    print(f"  {r['label']} | {r['num_segments']} segments | {r['chars']} chars | {r['audio_duration']:.1f}s")
    print(f"{'='*70}")
    print(r["text"])
    print()


  0_baseline | 111 segments | 2132 chars | 360.0s
 Остройството не е позиционирано правилно. Завъртете го така, че физическите бутони да бъдат отгоре. А това е ОК. Не е изглазило. А това е ОК. Не е изглазило. Ей, това е невалидно, нали? Да. Невалидно. 27. 27? Какво е това? Кой е също? 27. Това е невалидно, камера. Ай, това е невалидно, айте. Това е невалидно, камера. 27. Остройството не е позиционирано правилно. Завъртете го така, че физическите бутони да бъдат отгоре. 18 към 18. А, да. Това е. Тук в тази мигун... Ето ги дадете. Е, дай си се. Това е много хубаво. А 26 има. 26. 26 са при мен. Да. Така, 10 има ли още някъде? 10 са само излезти. 10, но само излезта и няма. Това е нацитата. Сега той трябва да се напиша. Аз ще пиша. Врачите, трябва да го почнем от най-малкото. Да. Добре, там, шкодите в раме. Устройството не е позиционирано правилно. Завъртете го така, че физическите бутони да бъдат отгоре. Трябва да сместваме. Не, не. Три, четири. Като са малко искате и преференциите да ка

## 16. Save Results & Recommendation

In [21]:
# Save all results to JSON for future reference
summary = {}
for key, r in results.items():
    summary[key] = {
        "label": r["label"],
        "chars": r["chars"],
        "num_segments": r["num_segments"],
        "audio_duration": r["audio_duration"],
        "api_time": r["api_time"],
        "text": r["text"],
    }

json_path = OUTPUT_DIR / "filtering_comparison.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f"Saved: {json_path}")

# List all generated WAV files
print(f"\nFiltered audio files in {OUTPUT_DIR}:")
for f in sorted(OUTPUT_DIR.glob("*.wav")):
    print(f"  {f.name:<45s} {f.stat().st_size / 1024:>7.1f} KB")

print(f"\nReview the full text outputs above and choose the technique")
print(f"with the most accurate Bulgarian transcription for the pipeline.")

Saved: /home/airsrg/work/video_streaming/output/filtering_comparison/filtering_comparison.json

Filtered audio files in /home/airsrg/work/video_streaming/output/filtering_comparison:
  election_6to12min.wav                         11250.1 KB
  filtered_0_baseline.wav                       11250.0 KB
  filtered_10_multiband_agc.wav                 11250.0 KB
  filtered_1_noise_reduction.wav                11250.0 KB
  filtered_2_vad_trimming.wav                    2630.5 KB
  filtered_3_loudness_norm.wav                  11250.0 KB
  filtered_4_highpass_deess.wav                 11250.0 KB
  filtered_5_compression.wav                    11250.0 KB
  filtered_6_bandpass_preemph.wav               11250.0 KB
  filtered_7_adaptive_vad.wav                   11250.0 KB
  filtered_8_agc.wav                            11250.0 KB
  filtered_9_boost_compress.wav                 11250.0 KB
  filtered_A_nr_norm.wav                        11250.0 KB
  filtered_B_studio.wav                         11